In [ ]:
# ============================================================
# HƯỚNG 4 - MỐC 3: HUẤN LUYỆN TSVM SEED 0 BẰNG KERNEL selective_scan (mamba_ssm), 100 EPOCH
# Settings: GPU T4 x2, Internet On (tải yolo26s-seg.pt). Input: h4-tsvm-cuda + h4-mamba-wheel.
# CELL 0: CÀI mamba_ssm TỪ WHEEL ĐÃ DỰNG Ở MỐC 2 (vài giây, không biên dịch lại)
# ============================================================
import os, sys, glob, time, json, shutil, subprocess
import torch
T_PHIEN = time.time()
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| Python", sys.version.split()[0])
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.__version__.startswith("2.11.0"):
    print("⚠️ Kaggle đã đổi torch (wheel dựng với 2.11.0+cu130). Nếu ô kiểm tra dưới báo lỗi undefined symbol, gửi lại mình.")

def da_co_kernel():
    r = subprocess.run([sys.executable, "-c", "import torch, selective_scan_cuda, mamba_ssm; "
                        "from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; print(mamba_ssm.__version__)"],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-1500:]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops"], check=False)
ok, msg = da_co_kernel()
if not ok:
    whl = sorted(glob.glob("/kaggle/input/**/mamba_ssm-*.whl", recursive=True))
    assert whl, "❌ Chưa thêm dataset h4-mamba-wheel (Add Input)"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", whl[0]], check=True)
    ok, msg = da_co_kernel()
print(msg)
assert ok, "❌ Wheel không nạp được kernel, gửi mình đoạn lỗi in phía trên"
print("✅ Có kernel selective_scan_cuda, mamba_ssm", msg.splitlines()[-1])


In [ ]:
# ============================================================
# CELL 1: NẠP ULTRALYTICS BẢN HƯỚNG 4, YAML SCALE s, DATA YAML (giống notebook báo cáo)
# ============================================================
import yaml
def tim(mau):
    return sorted(glob.glob(mau, recursive=True))
_repo = tim("/kaggle/input/**/ultralytics_TSVM_CUDA/PATCH_FP32.md")
_data = tim("/kaggle/input/**/Kvasir_YOLO_SEG_BG20/images/val")
assert _repo and _data, "❌ Chưa thêm dataset h4-tsvm-cuda (Add Input)"
CUSTOM_REPO_PATH = os.path.dirname(_repo[0])
DATASET_ROOT = os.path.dirname(os.path.dirname(_data[0]))
if not os.path.exists("/kaggle/working/ultralytics"):
    os.symlink(CUSTOM_REPO_PATH, "/kaggle/working/ultralytics")
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")
os.environ["TSVM_SCAN"] = "cuda"   # bắt buộc dùng kernel; thiếu kernel thì báo lỗi chứ không lặng lẽ quay về PyTorch
import ultralytics
import ultralytics.nn.modules.topology_shape_vmamba as tsv
assert ultralytics.__version__ == "8.4.127" and ultralytics.__file__.startswith("/kaggle/working/ultralytics")
assert tsv.cuda_kernel_available(), tsv._SELECTIVE_SCAN_IMPORT_ERROR
assert tsv.scan_backend() == "cuda"
print(f"✅ Ultralytics {ultralytics.__version__} (Hướng 4), TSVM_SCAN={tsv.scan_backend()}")

YAML_S = "/kaggle/working/yolo26s-seg-TopologyShapeVMamba.yaml"
with open(f"{CUSTOM_REPO_PATH}/cfg/models/26/yolo26-seg-TopologyShapeVMamba.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["scale"] = "s"
with open(YAML_S, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
dem = {s: len(os.listdir(f"{DATASET_ROOT}/images/{s}")) for s in ("train", "val")}
assert dem == {"train": 1040, "val": 160}, dem
DATA_YAML = "/kaggle/working/data_bg20.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": DATASET_ROOT, "train": "images/train", "val": "images/val", "nc": 1, "names": {0: "polyp"}}, f)
print("📁", DATASET_ROOT, dem)


In [ ]:
# ============================================================
# CELL 2: KHÓA TẤT ĐỊNH VÀ HUẤN LUYỆN (siêu tham số chép nguyên từ args.yaml lượt báo cáo TSVM_FP32 s0)
# ============================================================
import random, numpy as np
from ultralytics import YOLO
ARGS = {'epochs': 100, 'time': None, 'patience': 100, 'batch': 8, 'imgsz': 640, 'save': True, 'save_period': -1, 'cache': False, 'workers': 2, 'cls_remap': True, 'optimizer': 'AdamW', 'verbose': True, 'seed': 0, 'deterministic': True, 'single_cls': False, 'rect': False, 'cos_lr': False, 'close_mosaic': 10, 'resume': False, 'amp': False, 'fraction': 1.0, 'profile': False, 'freeze': None, 'multi_scale': 0.0, 'compile': False, 'channels_last': False, 'overlap_mask': True, 'mask_ratio': 4, 'dropout': 0.0, 'val': True, 'split': 'val', 'save_json': False, 'conf': None, 'iou': 0.7, 'max_det': 300, 'quantize': None, 'dnn': False, 'plots': True, 'end2end': None, 'source': None, 'vid_stride': 1, 'stream_buffer': False, 'visualize': False, 'augment': False, 'agnostic_nms': False, 'classes': None, 'retina_masks': False, 'embed': None, 'show': False, 'save_frames': False, 'save_txt': False, 'save_conf': False, 'save_crop': False, 'show_labels': True, 'show_conf': True, 'show_boxes': True, 'line_width': None, 'format': 'torchscript', 'keras': False, 'optimize': False, 'dynamic': False, 'simplify': True, 'opset': None, 'workspace': None, 'nms': False, 'lr0': 0.001, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 5.0, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1, 'distill_model': None, 'dis': 6.0, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'pose': 12.0, 'kobj': 1.0, 'rle': 1.0, 'angle': 1.0, 'dlog': 1.0, 'dgrad': 0.5, 'dlam': 1.0, 'nbs': 64, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'copy_paste_mode': 'flip', 'auto_augment': 'randaugment', 'erasing': 0.4, 'cfg': None, 'tracker': 'tracktrack.yaml'}
SEED = ARGS["seed"]
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)

TEN = f"Kvasir_BG20_YOLO26s_seg_TSVM_CUDA_s{SEED}_w{ARGS['workers']}"
model = YOLO(YAML_S).load("yolo26s-seg.pt")
assert type(model.model.model[10]).__name__ == "C2TSVMamba"
t0 = time.time()
model.train(data=DATA_YAML, device=0, project="/kaggle/working/runs", name=TEN, exist_ok=True, **ARGS)
GIO = (time.time() - t0) / 3600
print(f"⏱️ Huấn luyện {GIO:.2f} giờ")


In [ ]:
# ============================================================
# CELL 3: TỔNG HỢP, SO VỚI LƯỢT BÁO CÁO (PyTorch s0: 0,7245 tại epoch 88; 4,52 giờ), NÉN KẾT QUẢ
# ============================================================
import pandas as pd
RUN = f"/kaggle/working/runs/{TEN}"
d = pd.read_csv(f"{RUN}/results.csv"); d.columns = [c.strip() for c in d.columns]
cot = "metrics/mAP50-95(M)"
i = d[cot].idxmax()
tom_tat = {
    "ten_luot": TEN, "gio_huan_luyen": round(GIO, 3), "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__, "scan_backend": tsv.scan_backend(),
    "epoch_tot_nhat": int(d.loc[i, "epoch"]), "mask_map50_95_tot_nhat": float(d.loc[i, cot]),
    "mask_map50_tai_do": float(d.loc[i, "metrics/mAP50(M)"]), "box_map50_95_tai_do": float(d.loc[i, "metrics/mAP50-95(B)"]),
    "tham_chieu_pytorch_s0": {"mask_map50_95": 0.7245, "epoch": 88, "gio": 4.52},
}
tom_tat["delta_map"] = round(tom_tat["mask_map50_95_tot_nhat"] - 0.7245, 4)
tom_tat["ket_luan"] = ("tương đương (|Δ| ≤ 0,0074 = 1 SD 20 seed), dừng ở 1 seed"
                       if abs(tom_tat["delta_map"]) <= 0.0074 else "lệch hơn 1 SD, cần chạy thêm seed 1–4")
print(json.dumps(tom_tat, ensure_ascii=False, indent=1))
json.dump(tom_tat, open(f"{RUN}/tom_tat_moc3.json", "w"), ensure_ascii=False, indent=1)
shutil.make_archive("/kaggle/working/h4_moc3_s0", "zip", "/kaggle/working/runs", TEN)
print("Tải về: h4_moc3_s0.zip | tổng phiên", round((time.time() - T_PHIEN) / 3600, 2), "giờ")
